In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [8]:
BASE_DIR = Path().resolve()  # or adjust if needed
RAW_DATA_DIR = BASE_DIR / "raw_data"
CLEANED_DATA_DIR = BASE_DIR / "cleaned_data"

START_DATE = "2015-01-01"
END_DATE = "2026-05-01"

PRICE_FILES = {
    "egx30": "EGX 30 Historical Data.csv",
    "egx100": "EGX 100 Historical Data.csv",
    "gold": "GAU_EGP Historical Data.csv",
    "tmg": "T M G Holding Stock Price History.csv",
    "sodic": "SODIC Stock Price History.csv",
    "palm_hills": "Palm Hills Develop Stock Price History.csv",
}

TBILL_FILE = "Egypt 1-Year Bond Yield Historical Data.csv"

In [9]:
def clean_price_file(name: str, filepath: Path, start: str, end: str) -> pd.DataFrame:
    print(f"[{name}] Reading {filepath.name}")
    df = pd.read_csv(filepath)

    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    df.columns = [c.replace(".", "") for c in df.columns]
    df.columns = [c.replace("%", "pct") for c in df.columns]

    print(f"[{name}] Columns: {df.columns.tolist()}")

    df["date"] = pd.to_datetime(df["date"], format="%m/%d/%Y")
    df = df.set_index("date").sort_index()

    df = df.loc[start:end]

    for col in ["price", "open", "high", "low"]:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)
                .str.replace(",", "", regex=False)
                .str.strip()
            )
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "vol" in df.columns:
        df["vol"] = (
            df["vol"]
            .astype(str)
            .str.replace("M", "", regex=False)
            .str.strip()
            .replace("-", pd.NA)
        )
        df["vol"] = pd.to_numeric(df["vol"], errors="coerce")

    if "change_pct" in df.columns:
        df["change_pct"] = (
            df["change_pct"]
            .astype(str)
            .str.replace("%", "", regex=False)
            .str.strip()
        )
        df["change_pct"] = pd.to_numeric(df["change_pct"], errors="coerce")

    print(f"[{name}] Done. Shape: {df.shape}")
    return df

In [10]:
def clean_tbill_file(filepath: Path, start: str, end: str) -> pd.DataFrame:
    print(f"[tbills] Reading {filepath.name}")
    df = pd.read_csv(filepath)

    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

    date_col = df.columns[0]
    df = df.rename(columns={date_col: "date"})

    df = df[["date", "price"]]

    df["date"] = pd.to_datetime(df["date"], format="%m/%d/%Y")
    df = df.set_index("date").sort_index()

    df = df.loc[start:end]

    df["price"] = (
        df["price"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip()
    )
    df["price"] = pd.to_numeric(df["price"], errors="coerce")

    print(f"[tbills] Done. Shape: {df.shape}")
    return df

In [11]:
cleaned_data = {}

# Prices
for name, file in PRICE_FILES.items():
    path = RAW_DATA_DIR / file
    cleaned_data[name] = clean_price_file(name, path, START_DATE, END_DATE)

# T-bills
tbill_path = RAW_DATA_DIR / TBILL_FILE
cleaned_data["tbills"] = clean_tbill_file(tbill_path, START_DATE, END_DATE)

[egx30] Reading EGX 30 Historical Data.csv
[egx30] Columns: ['date', 'price', 'open', 'high', 'low', 'vol', 'change_pct']
[egx30] Done. Shape: (2754, 6)
[egx100] Reading EGX 100 Historical Data.csv
[egx100] Columns: ['date', 'price', 'open', 'high', 'low', 'vol', 'change_pct']
[egx100] Done. Shape: (1306, 6)
[gold] Reading GAU_EGP Historical Data.csv
[gold] Columns: ['date', 'price', 'open', 'high', 'low', 'vol', 'change_pct']
[gold] Done. Shape: (559, 6)
[tmg] Reading T M G Holding Stock Price History.csv
[tmg] Columns: ['date', 'price', 'open', 'high', 'low', 'vol', 'change_pct']
[tmg] Done. Shape: (2755, 6)
[sodic] Reading SODIC Stock Price History.csv
[sodic] Columns: ['date', 'price', 'open', 'high', 'low', 'vol', 'change_pct']
[sodic] Done. Shape: (2753, 6)
[palm_hills] Reading Palm Hills Develop Stock Price History.csv
[palm_hills] Columns: ['date', 'price', 'open', 'high', 'low', 'vol', 'change_pct']
[palm_hills] Done. Shape: (2755, 6)
[tbills] Reading Egypt 1-Year Bond Yield H

In [12]:
for name, df in cleaned_data.items():
    out_path = CLEANED_DATA_DIR / f"{name}.csv"
    df.to_csv(out_path)
    print(f"Saved {name} → {out_path}")

Saved egx30 → C:\Users\moata\Documents\Investments\cleaned_data\egx30.csv
Saved egx100 → C:\Users\moata\Documents\Investments\cleaned_data\egx100.csv
Saved gold → C:\Users\moata\Documents\Investments\cleaned_data\gold.csv
Saved tmg → C:\Users\moata\Documents\Investments\cleaned_data\tmg.csv
Saved sodic → C:\Users\moata\Documents\Investments\cleaned_data\sodic.csv
Saved palm_hills → C:\Users\moata\Documents\Investments\cleaned_data\palm_hills.csv
Saved tbills → C:\Users\moata\Documents\Investments\cleaned_data\tbills.csv
